In [1]:
import sys
import pandas as pd
from pathlib import Path

PROJECT_ROOT = Path(".").resolve().parent
sys.path.insert(0, str(PROJECT_ROOT))

INTERIM_DIR = PROJECT_ROOT / "data" / "processed"

from src.cleaning import (
    deduplicate,
    audit_nulls,
    drop_critical_nulls,
    cast_dtypes,
    normalize_cities,
    join_frames,
    print_schema_summary,
)

print("Environment ready.")

Environment ready.


In [2]:
reviews    = pd.read_parquet(INTERIM_DIR / "reviews_filtered.parquet")
businesses = pd.read_parquet(INTERIM_DIR / "businesses_filtered.parquet")
users      = pd.read_parquet(INTERIM_DIR / "users_filtered.parquet")

print(f"reviews    : {reviews.shape[0]:,} rows × {reviews.shape[1]} columns")
print(f"businesses : {businesses.shape[0]:,} rows × {businesses.shape[1]} columns")
print(f"users      : {users.shape[0]:,} rows × {users.shape[1]} columns")

reviews    : 381,999 rows × 9 columns
businesses : 8,203 rows × 7 columns
users      : 241,363 rows × 8 columns


In [3]:
reviews    = deduplicate(reviews,    key_col="review_id")
businesses = deduplicate(businesses, key_col="business_id")
users      = deduplicate(users,      key_col="user_id")

  Deduplication [review_id]: no duplicates found
  Deduplication [business_id]: no duplicates found
  Deduplication [user_id]: no duplicates found


In [4]:
print("--- reviews ---")
audit_nulls(reviews)

print("\n--- businesses ---")
audit_nulls(businesses)

print("\n--- users ---")
audit_nulls(users)

--- reviews ---
  Null audit: no missing values detected

--- businesses ---
  Null audit: no missing values detected

--- users ---
  Null audit: no missing values detected


,column,null_count,null_pct


In [5]:
# Drop any review records missing critical analysis fields
reviews = drop_critical_nulls(reviews, critical_cols=["text", "review_stars", "date"])
print(f"Reviews after critical-null drop: {len(reviews):,}")

  Dropped 0 rows with nulls in critical columns: ['text', 'review_stars', 'date']
Reviews after critical-null drop: 381,999


In [6]:
reviews    = cast_dtypes(reviews)
businesses = cast_dtypes(businesses)
users      = cast_dtypes(users)

print(f"\nreviews['date'] dtype   : {reviews['date'].dtype}")
print(f"reviews['review_stars'] dtype: {reviews['review_stars'].dtype}")

  Data type casting complete
  Data type casting complete
  Data type casting complete

reviews['date'] dtype   : datetime64[ns]
reviews['review_stars'] dtype: float64


In [7]:
businesses = normalize_cities(businesses)

print(f"Unique cities before normalization : {businesses['city'].nunique():,}")
print(f"Unique cities after normalization  : {businesses['city_normalized'].nunique():,}")

  City normalization complete — 508 unique cities
Unique cities before normalization : 520
Unique cities after normalization  : 508


In [8]:
df = join_frames(reviews, businesses, users)

print(f"\nJoined dataset: {df.shape[0]:,} rows × {df.shape[1]} columns")

  Three-way join complete → 381,999 rows × 19 columns

Joined dataset: 381,999 rows × 19 columns


In [9]:
print_schema_summary(df)


  Dataset shape: 381,999 rows × 19 columns


                                dtype  null_count  null_pct
review_id                      object           0       0.0
user_id                        object           0       0.0
business_id                    object           0       0.0
review_stars                  float64           0       0.0
date                   datetime64[ns]           0       0.0
text                           object           0       0.0
useful                          int64           0       0.0
funny                           int64           0       0.0
cool                            int64           0       0.0
name                           object           0       0.0
city                           object           0       0.0
city_normalized                object           0       0.0
state                          object           0       0.0
business_avg_stars            float64           0       0.0
business_review_count           int64           0       0.0
user_name                      object   

In [10]:
# First 5 rows — spot-check join correctness
for i, row in df.head(5).iterrows():
    print(f"--- Row {i} ---")
    for col in df.columns:
        val = str(row[col])
        if col == "text":
            val = val[:100]
        print(f"  {col}: {val}")
    print()

--- Row 0 ---
  review_id: -P5E9BYUaK7s3PwBF5oAyg
  user_id: Jha0USGDMefGFRLik_xFQg
  business_id: bMratNjTG5ZFEA6hVyr-xQ
  review_stars: 5.0
  date: 2017-02-19 00:00:00
  text: First time there and it was excellent!!! It feels like your are entering someone's home. The waiters
  useful: 0
  funny: 0
  cool: 0
  name: Portobello Cafe
  city: Eddystone
  city_normalized: Eddystone
  state: PA
  business_avg_stars: 4.0
  business_review_count: 137
  user_name: Lauren
  user_review_count: 116
  yelping_since: 2016-09-17 17:19:31
  user_avg_stars: 4.47

--- Row 1 ---
  review_id: A4n4YaE-owOVgTQcrVqHUw
  user_id: S7bjj-L07JuRr-tpX1UZLw
  business_id: I6L0Zxi5Ww0zEWSAVgngeQ
  review_stars: 4.0
  date: 2018-07-07 00:00:00
  text: The cafe was extremely cute. We came at 8am and they even had a jazz band playing at that time. I go
  useful: 0
  funny: 0
  cool: 0
  name: Cafe Beignet on Bourbon Street
  city: New Orleans
  city_normalized: New Orleans
  state: LA
  business_avg_stars: 3.5
  bu

In [11]:
output_path = INTERIM_DIR / "cleaned_joined.parquet"
df.to_parquet(output_path, index=False)

size_mb = output_path.stat().st_size / 1_048_576
print(f"Saved: {output_path.name}  ({size_mb:.1f} MB)")
print(f"Shape: {df.shape[0]:,} rows × {df.shape[1]} columns")

Saved: cleaned_joined.parquet  (139.0 MB)
Shape: 381,999 rows × 19 columns
